In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch_geometric.loader import DataLoader as GeoDataLoader
from torch_geometric.nn import global_mean_pool
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader


In [ ]:
import torch
import pickle
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import numpy as np
import pandas as pd
from numpy import savetxt, loadtxt
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.nn import Linear, BatchNorm1d
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import Input, Concatenate, Reshape, Conv1D, Flatten, Dense, Dropout, MultiHeadAttention, LayerNormalization, Add
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
from transformers import AutoTokenizer, AutoModel

from tqdm import tqdm
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, Dropout, Dense, concatenate, BatchNormalization



In [ ]:
def load_dataset(label_count):

    data = pd.read_csv("DATASET/multilabel_BILSTM_BERT.csv", index_col=False)
    data = data.drop(columns='Unnamed: 0')
    print(len(data))

    label_columns = ['Other', 'access_control', 'arithmetic', 'denial_service',
                     'front_running', 'reentrancy', 'time_manipulation',
                     'unchecked_low_calls']

    if label_count > len(label_columns):
        raise ValueError("Label count exceeds the available labels")

    selected_labels = label_columns[:label_count]

    for column in label_columns:
        data[column] = data[column].apply(lambda x: 1 if x != 0 else 0)

    # Giữ đúng 39645 rows
    #data = data.iloc[:39645]
    total_rows = len(data)
    split_point = int(0.8 * total_rows)

    print(split_point)
    print(total_rows - split_point)

    df_train = data.iloc[:split_point]
    df_test = data.iloc[split_point:]
    

    if True:
        X_train_source = np.load("DATASET/X_train_source_minmaxsummean_10chunks_512.npy")
        X_test_source = np.load("DATASET/X_test_source_minmaxsummean_10chunks_512.npy")
        X_train_source = X_train_source.reshape(-1, 10, 4, 768)
        X_test_source = X_test_source.reshape(-1, 10, 4, 768)
        X_train_source = X_train_source[:, :, 0, :]
        X_test_source = X_test_source[:, :, 0, :]
        X_train_source = X_train_source.reshape(-1, 10 * 768)
        X_test_source = X_test_source.reshape(-1, 10 * 768)

    else:

        # =========================
        # CODEBERT EMBEDDING
        # =========================

        source_train = df_train['sourcecode'].tolist()
        source_test = df_test['sourcecode'].tolist()
        

        tokenizer = AutoTokenizer.from_pretrained("./codebert")
        model = AutoModel.from_pretrained("./codebert")

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device)


        model.eval()

        print("Generating CodeBERT embeddings...")

        X_train_source = get_codebert_embeddings(source_train, tokenizer, model, device)
        X_test_source = get_codebert_embeddings(source_test, tokenizer, model, device)
        
            
        # texts = df_train['sourcecode'].astype(str).tolist()

        # lengths = [len(tokenizer.tokenize(x)) for x in texts]

        # print("Average tokens:", np.mean(lengths))
        # print("Max tokens:", np.max(lengths))


        print("Load Features CodeBERT success!!")
        np.save("DATASET/X_train_source.npy", X_train_source)
        np.save("DATASET/X_test_source.npy", X_test_source)

    # =========================
    # OPCODE FEATURES (giữ nguyên)
    # =========================

    X_train_opcode = np.array(df_train.iloc[:, 11:].values)
    X_test_opcode = np.array(df_test.iloc[:, 11:].values)

    # =========================
    # LABELS
    # =========================

    y_train = df_train[selected_labels].values
    y_test = df_test[selected_labels].values
    labels = data[selected_labels].values

    print(f"Load Data for {label_count}-Label MultiLabel success!!")

    return X_train_opcode, X_test_opcode, X_train_source, X_test_source, y_train, y_test, labels

In [ ]:
X_train_opcode, X_test_opcode, X_train_source, X_test_source, y_train, y_test, labels = load_dataset(8)

In [ ]:
import torch
import pickle
import numpy as np
import torch.nn.functional as F

from torch.nn import Linear, BatchNorm1d
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool
from sklearn.metrics import precision_score, recall_score, f1_score

# =========================
# LOAD DATA
# =========================
input_file = 'DATASET/gnn_input.pkl'

with open(input_file, 'rb') as f:
    raw_data = pickle.load(f)

feature_list = raw_data['features']
adj_matrices = raw_data['adj_matrices']
label_list = raw_data['labels']

target_dim = 57
num_classes = len(label_list[0])

def pad_or_truncate_features(features, target_dim):
    if features.shape[1] > target_dim:
        return features[:, :target_dim]
    elif features.shape[1] < target_dim:
        padded_features = np.zeros((features.shape[0], target_dim), dtype=np.float32)
        padded_features[:, :features.shape[1]] = features
        return padded_features
    else:
        return features.astype(np.float32)

def create_pyg_data(features, adj_matrix, label, target_dim):
    adj = torch.tensor(np.array(adj_matrix), dtype=torch.long)
    edge_index = adj.nonzero(as_tuple=False).t().contiguous()

    x = torch.tensor(pad_or_truncate_features(features, target_dim), dtype=torch.float)
    y = torch.tensor(label, dtype=torch.float)

    return Data(x=x, edge_index=edge_index, y=y)

data_list = [
    create_pyg_data(feat, adj, label, target_dim)
    for feat, adj, label in zip(feature_list, adj_matrices, label_list)
]

# nếu muốn shuffle trước khi split thì mở 2 dòng dưới
# indices = np.random.permutation(len(data_list))
# data_list = [data_list[i] for i in indices]

split_idx = int(len(data_list) * 0.8)
cfg_train_loader = DataLoader(data_list[:split_idx], batch_size=64, shuffle=False)
cfg_test_loader = DataLoader(data_list[split_idx:], batch_size=64, shuffle=False)


In [ ]:

class OpcodeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)     # input cho Embedding
        self.y = torch.tensor(y, dtype=torch.float32)  # cho BCELoss

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
class SourceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
opcode_train_dataset = OpcodeDataset(X_train_opcode, y_train)
opcode_test_dataset = OpcodeDataset(X_test_opcode, y_test)

opcode_train_loader = DataLoader(opcode_train_dataset, batch_size=64, shuffle=False)
opcode_test_loader  = DataLoader(opcode_test_dataset, batch_size=64, shuffle=False)

source_train_dataset = SourceDataset(X_train_source, y_train)
source_test_dataset  = SourceDataset(X_test_source, y_test)

source_train_loader = DataLoader(source_train_dataset, batch_size=64, shuffle=False)
source_test_loader  = DataLoader(source_test_dataset, batch_size=64, shuffle=False)

In [ ]:

class BiLSTMModel(nn.Module):
    def __init__(
        self,
        vocab_size=2000,
        embedding_dim=384,
        hidden_dim=128,
        output_dim=8
    ):
        super(BiLSTMModel, self).__init__()
        
        # Embedding
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Stacked BiLSTM (2 layers)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=2,              # 🔥 2 layer BiLSTM
            batch_first=True,
            bidirectional=True,
            dropout=0.3               # dropout giữa các layer LSTM
        )
        
        # Regularization
        self.dropout = nn.Dropout(0.3)
        self.bn1 = nn.BatchNorm1d(hidden_dim * 2)
        
        # Dense layers
        self.fc1 = nn.Linear(hidden_dim * 2, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.fc2 = nn.Linear(128, 64)
        
        # Output
        self.out = nn.Linear(64, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (B, 280)
        x = self.embedding(x)  # -> (B, 280, 286)
        
        # LSTM
        _, (h_n, _) = self.lstm(x)
        
        # Lấy hidden state cuối của 2 chiều (forward + backward)
        x = torch.cat((h_n[-2], h_n[-1]), dim=1)  # (B, 256)
        
        # Dense
        x = self.dropout(x)
        x = self.bn1(x)
        x = self.dropout(x)
        
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.bn2(x)
        x = self.dropout(x)
        
        x = self.fc2(x)
        x = torch.relu(x)
        return x
        
        x = self.out(x)
        x = self.sigmoid(x)
        
        return x

import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool

class GAT(torch.nn.Module):
    def __init__(self, in_channels=57, hidden_channels=64, num_classes=8):
        super().__init__()

        # ===== 2 GAT layers =====
        self.conv1 = GATConv(in_channels, hidden_channels, heads=4)
        self.conv2 = GATConv(hidden_channels * 4, hidden_channels, heads=2)

        # ===== Feature layer (64) =====
        self.fc1 = torch.nn.Linear(hidden_channels * 2, 64)

        # ===== Output =====
        self.lin = torch.nn.Linear(64, num_classes)

        self.dropout = 0.3

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # ===== Layer 1 =====
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        # ===== Layer 2 =====
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        # ===== Pooling =====
        x = global_mean_pool(x, batch)   # (B, hidden_channels)
        x = F.dropout(x, p=self.dropout, training=self.training)

        # ===== Feature 64 =====
        x = self.fc1(x)
        x = F.relu(x)   # (B, 64)

        return x
    
        # ===== Output =====
        x = self.lin(x)
        x = torch.sigmoid(x)
        return x

In [ ]:
codebert_model = load_model('DATASET/bert.h5')
codebert_model = Model(
    inputs=codebert_model.input,
    outputs=codebert_model.layers[-2].output   # layer 64
)

bilstm_model = BiLSTMModel()
bilstm_model.load_state_dict(torch.load("DATASET/bilstm_model.pth"))
device = "cuda" if torch.cuda.is_available() else "cpu"
bilstm_model.to(device)
bilstm_model.eval()

gat_model = GAT()
gat_model.load_state_dict(torch.load("DATASET/gat.pth"))
gat_model.to(device)
gat_model.eval()

In [ ]:
for i, layer in enumerate(codebert_model.layers):
    try:
        print(i, layer.name, layer.output_shape)
    except:
        print(i, layer.name, "no shape")

In [ ]:
class MetaClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(192, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 8),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
def extract_opcode_features(model, loader, device):
    model.eval()
    outputs = []

    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            out = model(x)   # (B, 8)
            outputs.append(out.cpu())

    return torch.cat(outputs, dim=0)  # (N, 8)


def extract_gat_features(model, loader, device):
    model.eval()
    outputs = []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data)   # (B, 8)
            outputs.append(out.cpu())

    return torch.cat(outputs, dim=0)


codebert_features_train = codebert_model.predict(X_train_source, batch_size=64)
codebert_features_test  = codebert_model.predict(X_test_source, batch_size=64)

opcode_feat_train = extract_opcode_features(bilstm_model, opcode_train_loader, device)
opcode_feat_test  = extract_opcode_features(bilstm_model, opcode_test_loader, device)

gat_feat_train = extract_gat_features(gat_model, cfg_train_loader, device)
gat_feat_test  = extract_gat_features(gat_model, cfg_test_loader, device)

# CodeBERT (Keras)
codebert_feat_train = codebert_model.predict(X_train_source, batch_size=64)
codebert_feat_test  = codebert_model.predict(X_test_source, batch_size=64)

# convert numpy -> torch nếu cần
codebert_feat_train = torch.tensor(codebert_feat_train)
codebert_feat_test  = torch.tensor(codebert_feat_test)

train_features = torch.cat([
    opcode_feat_train,
    gat_feat_train,
    codebert_feat_train
], dim=1)   # (N, 24)

test_features = torch.cat([
    opcode_feat_test,
    gat_feat_test,
    codebert_feat_test
], dim=1)

In [ ]:
class HierarchicalFusion(nn.Module):
    def __init__(self):
        super().__init__()

        self.gate = nn.Sequential(
            nn.Linear(64*3, 64*3),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.Linear(64*3, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 8)
        )

    def forward(self, opcode, gat, codebert):
        fused = torch.cat([opcode, gat, codebert], dim=1)

        gate = self.gate(fused)
        fused = fused * gate   # gating

        return self.classifier(fused)

In [ ]:
class FusionDataset(Dataset):
    def __init__(self, opcode, gat, codebert, y):
        self.opcode = opcode.float()
        self.gat = gat.float()
        self.codebert = codebert.float()
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.opcode[idx],
            self.gat[idx],
            self.codebert[idx],
            self.y[idx]
        )

In [ ]:
train_dataset = FusionDataset(
    opcode_feat_train,
    gat_feat_train,
    codebert_feat_train,
    y_train
)

test_dataset = FusionDataset(
    opcode_feat_test,
    gat_feat_test,
    codebert_feat_test,
    y_test
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
def train_model(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for opcode, gat, codebert, y in train_loader:
        opcode = opcode.to(device)
        gat = gat.to(device)
        codebert = codebert.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(opcode, gat, codebert)

        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

def evaluate_model(model, test_loader, device):
    model.eval()

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for opcode, gat, codebert, y in test_loader:

            opcode = opcode.to(device)
            gat = gat.to(device)
            codebert = codebert.to(device)

            outputs = model(opcode, gat, codebert)

            all_preds.append(outputs.cpu())
            all_targets.append(y)

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()

    # sigmoid for evaluation
    all_preds = 1 / (1 + np.exp(-all_preds))

    preds_bin = (all_preds > 0.5).astype(int)

    print("\n===== CLASSIFICATION REPORT =====")
    print(classification_report(all_targets, preds_bin, digits=4))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = HierarchicalFusion().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 1000

for epoch in range(EPOCHS):
    loss = train_model(model, train_loader, optimizer, criterion, device)

    

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {loss:.4f}")
    if (epoch + 1) % 100 == 0:
        evaluate_model(model, test_loader, device)